# DocRestore AI: Document Shadow Removal Using Pix2Pix GAN
### Google Colab GPU Training Pipeline

This notebook trains the **Pix2Pix Conditional GAN** (U-Net Generator + PatchGAN Discriminator) to restore document images affected by harsh cast shadows, folds, and non-uniform illumination.

**Architecture Highlights:**
- **Generator**: U-Net 256 with symmetrical skip connections preserving text strokes.
- **Discriminator**: 70x70 PatchGAN penalizing local high-frequency illumination errors.
- **Objective**: Combined Adversarial Loss + $\lambda L_1$ Loss ($\lambda = 100$).
- **Output**: Generates  for direct use in the local Streamlit application.

## Step 1: Verify GPU Environment
Ensure you are using a GPU runtime (**Runtime > Change runtime type > T4 GPU**).

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## Step 2: Download or Mount Paired Document Shadow Dataset
Standard benchmark datasets for document shadow removal include:
1. **Jung et al. Document Shadow Dataset** (paired shadow and shadow-free document scans)
2. **DocShadow / SDSR Dataset**

Run the cell below to set up training directories or load your dataset from Google Drive.

In [ ]:
import os
from pathlib import Path

# Training directory paths
TRAIN_SHADOW_DIR = "data/train/shadow"
TRAIN_CLEAN_DIR = "data/train/clean"
os.makedirs(TRAIN_SHADOW_DIR, exist_ok=True)
os.makedirs(TRAIN_CLEAN_DIR, exist_ok=True)

# Option A: Mount Google Drive if your dataset is in Drive
# from google.colab import drive
# drive.mount("/content/drive")
# !unzip /content/drive/MyDrive/document_shadow_dataset.zip -d data/

print("Data directories ready.")

## Step 3: Define Exact Pix2Pix Architecture
*(Strictly identical to local )*

In [ ]:
import torch
import torch.nn as nn

class UNetDownBlock(nn.Module):
    def __init__(self, in_c, out_c, normalize=True):
        super().__init__()
        layers = [nn.Conv2d(in_c, out_c, kernel_size=4, stride=2, padding=1, bias=not normalize)]
        if normalize:
            layers.append(nn.BatchNorm2d(out_c))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.block = nn.Sequential(*layers)
    def forward(self, x):
        return self.block(x)

class UNetUpBlock(nn.Module):
    def __init__(self, in_c, out_c, dropout=False):
        super().__init__()
        layers = [
            nn.ConvTranspose2d(in_c, out_c, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(out_c)
        ]
        if dropout:
            layers.append(nn.Dropout(0.5))
        layers.append(nn.ReLU(inplace=True))
        self.block = nn.Sequential(*layers)
    def forward(self, x, skip):
        x = self.block(x)
        return torch.cat([x, skip], dim=1)

class UNetGenerator(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, nf=64):
        super().__init__()
        self.e1 = UNetDownBlock(in_channels, nf, normalize=False)
        self.e2 = UNetDownBlock(nf, nf * 2, normalize=True)
        self.e3 = UNetDownBlock(nf * 2, nf * 4, normalize=True)
        self.e4 = UNetDownBlock(nf * 4, nf * 8, normalize=True)
        self.e5 = UNetDownBlock(nf * 8, nf * 8, normalize=True)
        self.e6 = UNetDownBlock(nf * 8, nf * 8, normalize=True)
        self.e7 = UNetDownBlock(nf * 8, nf * 8, normalize=True)
        self.e8 = nn.Sequential(
            nn.Conv2d(nf * 8, nf * 8, kernel_size=4, stride=2, padding=1, bias=False),
            nn.ReLU(inplace=True)
        )
        self.d1 = UNetUpBlock(nf * 8, nf * 8, dropout=True)
        self.d2 = UNetUpBlock(nf * 16, nf * 8, dropout=True)
        self.d3 = UNetUpBlock(nf * 16, nf * 8, dropout=True)
        self.d4 = UNetUpBlock(nf * 16, nf * 8, dropout=False)
        self.d5 = UNetUpBlock(nf * 16, nf * 4, dropout=False)
        self.d6 = UNetUpBlock(nf * 8, nf * 2, dropout=False)
        self.d7 = UNetUpBlock(nf * 4, nf, dropout=False)
        self.final = nn.Sequential(
            nn.ConvTranspose2d(nf * 2, out_channels, kernel_size=4, stride=2, padding=1),
            nn.Tanh()
        )
    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(e1)
        e3 = self.e3(e2)
        e4 = self.e4(e3)
        e5 = self.e5(e4)
        e6 = self.e6(e5)
        e7 = self.e7(e6)
        e8 = self.e8(e7)
        d1 = self.d1(e8, e7)
        d2 = self.d2(d1, e6)
        d3 = self.d3(d2, e5)
        d4 = self.d4(d3, e4)
        d5 = self.d5(d4, e3)
        d6 = self.d6(d5, e2)
        d7 = self.d7(d6, e1)
        return self.final(d7)

class PatchGANDiscriminator(nn.Module):
    def __init__(self, in_channels=6, nf=64):
        super().__init__()
        def block(in_c, out_c, stride=2, normalize=True):
            layers = [nn.Conv2d(in_c, out_c, kernel_size=4, stride=stride, padding=1, bias=not normalize)]
            if normalize:
                layers.append(nn.BatchNorm2d(out_c))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers
        self.model = nn.Sequential(
            *block(in_channels, nf, stride=2, normalize=False),
            *block(nf, nf * 2, stride=2, normalize=True),
            *block(nf * 2, nf * 4, stride=2, normalize=True),
            *block(nf * 4, nf * 8, stride=1, normalize=True),
            nn.Conv2d(nf * 8, 1, kernel_size=4, stride=1, padding=1)
        )
    def forward(self, condition, candidate):
        return self.model(torch.cat([condition, candidate], dim=1))

print("Pix2Pix Architecture initialized successfully.")

## Step 4: Run Training Loop
Trains for 50-80 epochs. Saves  and .

In [ ]:
# Hyperparameters
EPOCHS = 80
BATCH_SIZE = 16
LR = 0.0002
LAMBDA_L1 = 100.0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on {DEVICE} for {EPOCHS} epochs.")
# (Run training function as provided in colab/train_pix2pix.py)

## Step 5: Download Trained Checkpoint for Local Streamlit App
Download  and place it in your local  folder.

In [ ]:
from google.colab import files
# Download the trained model
# files.download("checkpoints/generator.pth")